# Startup.OS — Performance Dashboard & Business Intelligence
**A Python showcase of the [Startup.OS](https://startup-pulse-peek.lovable.app/) dashboard**

This notebook covers:
1. Core KPI dashboard (revenue, profit, burn, CAC)
2. Financial Health (runway, break-even, growth rates)
3. Customer Intelligence (LTV:CAC, payback, churn, cohort retention)
4. Revenue Quality (MRR/ARR, NRR, concentration risk)
5. Operational Efficiency (gross margin, expense breakdown, headcount)
6. Forecasting (3-scenario, Monte Carlo burn, CAC extrapolation)

> All figures are simulated for demo purposes.


## 0. Setup

In [ ]:
# Install dependencies (run once)
# !pip install matplotlib numpy rich

import matplotlib
matplotlib.use('Agg')  # Remove this line if running interactively
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import random
from datetime import datetime, timedelta
from collections import defaultdict

print("✅ Dependencies loaded")

---
## 1. Data Simulation
All business data is generated here — swap in your real data to use this notebook in production.

In [ ]:
PRODUCTS = [
    {"name": "Filly Suite",     "color": "#6EE7B7"},
    {"name": "Aegis Security",  "color": "#93C5FD"},
    {"name": "Filly API",       "color": "#FCA5A5"},
    {"name": "Other",           "color": "#D1D5DB"},
]

CUSTOMERS = [
    "Northwind Labs", "Helio Studio", "Vector Capital",
    "Quantum Forge",  "Lumen Works",  "Ridge & Co.",
    "Apex Dynamics",  "Nova Systems", "Drift Analytics",
    "Prism Ventures",
]

STATUSES = ["paid", "paid", "paid", "pending", "failed"]

def generate_metrics():
    return {
        "revenue_ytd":    1_160_500,
        "revenue_mom":      178_400,
        "revenue_growth":      18.4,
        "profit_ytd":      638_800,
        "profit_margin":      55.0,
        "profit_growth":      32.1,
        "expenses_ytd":    521_700,
        "expenses_growth":     6.2,
        "cac":                 156,
        "cac_growth":          9.3,
        "as_of": datetime.utcnow().strftime("%b %d, %H:%M UTC"),
    }

def generate_monthly_series(months=12):
    base_revenue  = 70_000
    base_expenses = 40_000
    base_cac      = 130
    series = []
    date = datetime.utcnow().replace(day=1) - timedelta(days=365)
    for i in range(months):
        revenue  = int(base_revenue  * (1 + i * 0.04) * random.uniform(0.95, 1.12))
        expenses = int(base_expenses * (1 + i * 0.02) * random.uniform(0.97, 1.05))
        cac      = round(base_cac + i * 2.2 + random.uniform(-5, 5), 2)
        series.append({
            "month":    date.strftime("%b %Y"),
            "revenue":  revenue,
            "expenses": expenses,
            "profit":   revenue - expenses,
            "cac":      cac,
        })
        date = (date.replace(day=28) + timedelta(days=4)).replace(day=1)
    return series

def generate_product_revenue():
    totals = [78_400, 52_100, 31_200, 16_700]
    total  = sum(totals)
    return [
        {**p, "amount": a, "share": round(a / total * 100)}
        for p, a in zip(PRODUCTS, totals)
    ]

def generate_transactions(n=50):
    prices    = [4_800, 1_290, 890, 2_400, 690, 4_800, 3_200, 1_100, 560, 2_950]
    tx_id     = 9241
    result    = []
    for i in range(n):
        product = random.choice([p["name"] for p in PRODUCTS[:-1]])
        tier    = random.choice(["", " Pro", " Enterprise", " Standard"])
        result.append({
            "id":       f"TX-{tx_id - i}",
            "customer": CUSTOMERS[i % len(CUSTOMERS)],
            "product":  product + tier,
            "amount":   prices[i % len(prices)],
            "status":   random.choice(STATUSES),
        })
    return result

# Generate all data
metrics      = generate_metrics()
monthly      = generate_monthly_series()
products     = generate_product_revenue()
transactions = generate_transactions()

print(f"✅ Data generated — {len(monthly)} months, {len(transactions)} transactions")
print(f"   Revenue YTD: ${metrics['revenue_ytd']:,}  |  Net Profit: ${metrics['profit_ytd']:,}")

---
## 2. Chart Theme

In [ ]:
BG      = "#0D0F14"
SURFACE = "#161920"
ACCENT1 = "#6EE7B7"   # teal
ACCENT2 = "#F87171"   # coral
ACCENT3 = "#93C5FD"   # blue
ACCENT4 = "#FCD34D"   # amber
MUTED   = "#6B7280"
TEXT    = "#E5E7EB"

plt.rcParams.update({
    "figure.facecolor":  BG,
    "axes.facecolor":    SURFACE,
    "axes.edgecolor":    MUTED,
    "axes.labelcolor":   TEXT,
    "xtick.color":       MUTED,
    "ytick.color":       MUTED,
    "text.color":        TEXT,
    "grid.color":        "#1F2937",
    "grid.linestyle":    "--",
    "grid.linewidth":    0.6,
    "font.family":       "monospace",
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

print("✅ Chart theme applied")

---
## 3. Core Dashboard Charts

### 3.1 Revenue vs Expenses — Last 12 Months

In [ ]:
labels   = [m["month"]    for m in monthly]
revenue  = [m["revenue"]  / 1_000 for m in monthly]
expenses = [m["expenses"] / 1_000 for m in monthly]
x = np.arange(len(labels))
w = 0.38

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w/2, revenue,  w, color=ACCENT1, alpha=0.9, label="Revenue")
ax.bar(x + w/2, expenses, w, color=ACCENT2, alpha=0.9, label="Expenses")
ax.set_title("Revenue vs Expenses — Last 12 Months (USD '000)", fontsize=13, color=TEXT, pad=16)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("USD (thousands)")
ax.yaxis.grid(True); ax.set_axisbelow(True); ax.legend(framealpha=0)
plt.tight_layout(); plt.show()

### 3.2 Revenue by Product — Donut Chart

In [ ]:
labels  = [p["name"]   for p in products]
amounts = [p["amount"] for p in products]
colors  = [ACCENT1, ACCENT3, ACCENT2, MUTED]

fig, ax = plt.subplots(figsize=(7, 7))
ax.set_facecolor(BG); fig.patch.set_facecolor(BG)
wedges, texts, autotexts = ax.pie(
    amounts, labels=labels, colors=colors, autopct="%1.0f%%",
    startangle=90, pctdistance=0.78,
    wedgeprops={"width": 0.52, "edgecolor": BG, "linewidth": 2},
)
for t in texts:     t.set_color(TEXT); t.set_fontsize(11)
for at in autotexts: at.set_color(BG); at.set_fontsize(9); at.set_fontweight("bold")
ax.set_title("Revenue by Product — Current Quarter", fontsize=13, color=TEXT, pad=20)
ax.text(0, 0, f"${sum(amounts)/1_000:.0f}K", ha="center", va="center",
        fontsize=20, fontweight="bold", color=TEXT)
plt.tight_layout(); plt.show()

### 3.3 Net Profit — Monthly

In [ ]:
profit = [m["profit"] / 1_000 for m in monthly]
x = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(x, profit, alpha=0.18, color=ACCENT3)
ax.plot(x, profit, color=ACCENT3, linewidth=2.2, marker="o", markersize=5, markerfacecolor=BG)
ax.set_title("Net Profit — Monthly Contribution (USD '000)", fontsize=13, color=TEXT, pad=16)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("USD (thousands)")
ax.yaxis.grid(True); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

### 3.4 Customer Acquisition Cost — 12-Month Trend

In [ ]:
cac_vals = [m["cac"] for m in monthly]

fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(x, cac_vals, alpha=0.15, color=ACCENT4)
ax.plot(x, cac_vals, color=ACCENT4, linewidth=2.2, marker="s", markersize=5, markerfacecolor=BG)
ax.set_title("Customer Acquisition Cost — 12-Month Trend (USD)", fontsize=13, color=TEXT, pad=16)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("CAC (USD)")
ax.yaxis.grid(True); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

---
## 4. Financial Health

### 4.1 Runway Calculator

In [ ]:
CASH_BALANCE  = 2_400_000
MONTHLY_BURN  = 43_475

months_runway = CASH_BALANCE / MONTHLY_BURN
safe_until    = (datetime.utcnow() + timedelta(days=months_runway * 30)).strftime("%b %Y")
status        = "CRITICAL" if months_runway < 6 else "CAUTION" if months_runway < 12 else "HEALTHY"

print(f"💰 Runway Analysis")
print(f"   Cash on hand : ${CASH_BALANCE:,}")
print(f"   Monthly burn : ${MONTHLY_BURN:,}")
print(f"   Runway       : {months_runway:.1f} months")
print(f"   Safe until   : {safe_until}")
print(f"   Status       : {status}")

### 4.2 Break-Even Analysis

In [ ]:
FIXED_COSTS          = 30_000
VARIABLE_COST_PCT    = 0.28
AVG_REV_PER_CUSTOMER = 1_290

contribution_margin = AVG_REV_PER_CUSTOMER * (1 - VARIABLE_COST_PCT)
units_needed        = FIXED_COSTS / contribution_margin
revenue_needed      = units_needed * AVG_REV_PER_CUSTOMER

print(f"⚖️  Break-Even Analysis")
print(f"   Contribution margin : ${contribution_margin:,.0f} ({(1-VARIABLE_COST_PCT)*100:.0f}%)")
print(f"   Customers needed    : {units_needed:.0f}")
print(f"   Revenue needed      : ${revenue_needed:,.0f}")

### 4.3 MoM & QoQ Growth Rates

In [ ]:
growth_data = []
for i, m in enumerate(monthly):
    mom = round((m["revenue"] - monthly[i-1]["revenue"]) / monthly[i-1]["revenue"] * 100, 1) if i >= 1 else None
    qoq = round((m["revenue"] - monthly[i-3]["revenue"]) / monthly[i-3]["revenue"] * 100, 1) if i >= 3 else None
    growth_data.append({"month": m["month"], "revenue": m["revenue"], "mom": mom, "qoq": qoq})

print(f"{'Month':<12} {'Revenue':>10} {'MoM %':>8} {'QoQ %':>8}")
print("-" * 42)
for r in growth_data:
    mom_str = f"{r['mom']:+.1f}%" if r["mom"] is not None else "  —"
    qoq_str = f"{r['qoq']:+.1f}%" if r["qoq"] is not None else "  —"
    print(f"{r['month']:<12} ${r['revenue']:>9,} {mom_str:>8} {qoq_str:>8}")

---
## 5. Customer Intelligence

### 5.1 LTV : CAC Ratio

In [ ]:
GROSS_MARGIN_PCT  = 0.72
MONTHLY_CHURN     = 0.025
CAC               = metrics["cac"]

avg_lifetime_months = 1 / MONTHLY_CHURN
ltv   = AVG_REV_PER_CUSTOMER * GROSS_MARGIN_PCT * avg_lifetime_months
ratio = ltv / CAC
health = "EXCELLENT" if ratio >= 5 else "GOOD" if ratio >= 3 else "WEAK" if ratio >= 1 else "POOR"

print(f"📈 LTV : CAC Analysis")
print(f"   Avg customer lifetime : {avg_lifetime_months:.1f} months")
print(f"   LTV                   : ${ltv:,.0f}")
print(f"   CAC                   : ${CAC:,}")
print(f"   LTV:CAC Ratio         : {ratio:.1f}x  [{health}]")
print(f"   Benchmark             : > 3x healthy, > 5x excellent")

# Visualise
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(["CAC", "LTV"], [CAC, ltv], color=[ACCENT2, ACCENT1], alpha=0.85, height=0.4)
ax.axvline(CAC * 3, color=ACCENT4, linewidth=1.5, linestyle="--", label="3x CAC benchmark")
for bar, val in zip(bars, [CAC, ltv]):
    ax.text(val + 50, bar.get_y() + bar.get_height()/2, f"${val:,.0f}", va="center", color=TEXT)
ax.set_title(f"LTV : CAC  =  {ratio:.1f}x  ({health})", fontsize=13, color=TEXT, pad=16)
ax.xaxis.grid(True); ax.set_axisbelow(True); ax.legend(framealpha=0)
plt.tight_layout(); plt.show()

### 5.2 Payback Period

In [ ]:
monthly_gp     = AVG_REV_PER_CUSTOMER * GROSS_MARGIN_PCT
payback_months = CAC / monthly_gp
pb_health      = "EXCELLENT" if payback_months < 12 else "GOOD" if payback_months < 18 else "CAUTION" if payback_months < 24 else "POOR"

print(f"⏱️  Payback Period")
print(f"   Monthly gross profit : ${monthly_gp:,.0f}")
print(f"   CAC                  : ${CAC:,}")
print(f"   Payback period       : {payback_months:.1f} months  [{pb_health}]")

### 5.3 Churn Analysis

In [ ]:
annual_churn = 1 - (1 - MONTHLY_CHURN) ** 12
arr_at_risk  = metrics["revenue_ytd"] * annual_churn

print(f"📉 Churn Analysis")
print(f"   Monthly churn rate : {MONTHLY_CHURN*100:.1f}%")
print(f"   Annual churn rate  : {annual_churn*100:.1f}%")
print(f"   ARR at risk        : ${arr_at_risk:,.0f}")

### 5.4 Cohort Retention Curve

In [ ]:
INITIAL_CUSTOMERS = 200
cohort = []
remaining = INITIAL_CUSTOMERS
for i in range(13):
    churned   = round(remaining * MONTHLY_CHURN) if i > 0 else 0
    remaining = max(0, remaining - churned)
    cohort.append({"month": i, "customers": remaining, "retention": round(remaining / INITIAL_CUSTOMERS * 100, 1)})

fig, ax = plt.subplots(figsize=(10, 5))
m_vals = [r["month"]     for r in cohort]
r_vals = [r["retention"] for r in cohort]
ax.fill_between(m_vals, r_vals, alpha=0.18, color=ACCENT1)
ax.plot(m_vals, r_vals, color=ACCENT1, linewidth=2.2, marker="o", markersize=5, markerfacecolor=BG)
ax.axhline(50, color=MUTED, linewidth=0.8, linestyle=":")
ax.set_title("Cohort Retention Curve", fontsize=13, color=TEXT, pad=16)
ax.set_xlabel("Month"); ax.set_ylabel("Customers Retained (%)")
ax.yaxis.grid(True); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

---
## 6. Revenue Quality

### 6.1 MRR / ARR

In [ ]:
mrr = monthly[-1]["revenue"]
arr = mrr * 12
print(f"📊 MRR / ARR")
print(f"   MRR : ${mrr:,}")
print(f"   ARR : ${arr:,}  (${arr/1_000_000:.2f}M)")

### 6.2 Net Revenue Retention (NRR)

In [ ]:
NRR_EXPANSION   = 8_500
NRR_CONTRACTION = 2_200
NRR_CHURN_MRR   = 3_800

nrr = (mrr + NRR_EXPANSION - NRR_CONTRACTION - NRR_CHURN_MRR) / mrr * 100
nrr_health = "EXCELLENT" if nrr >= 120 else "GOOD" if nrr >= 100 else "CAUTION" if nrr >= 85 else "POOR"

print(f"🔄 Net Revenue Retention")
print(f"   Base MRR    : ${mrr:,}")
print(f"   + Expansion : ${NRR_EXPANSION:,}")
print(f"   - Churn MRR : ${NRR_CHURN_MRR:,}")
print(f"   NRR         : {nrr:.1f}%  [{nrr_health}]")
print(f"   Benchmark   : > 100% = growing without new customers")

### 6.3 Revenue Concentration Risk

In [ ]:
customer_rev = defaultdict(float)
for tx in transactions:
    if tx["status"] == "paid":
        customer_rev[tx["customer"]] += tx["amount"]

total    = sum(customer_rev.values())
sorted_c = sorted(customer_rev.items(), key=lambda x: x[1], reverse=True)
top3     = sorted_c[:3]
top3_rev = sum(v for _, v in top3)
conc_pct = top3_rev / total * 100
risk     = "HIGH" if conc_pct > 50 else "MEDIUM" if conc_pct > 35 else "LOW"

print(f"⚠️  Revenue Concentration")
print(f"   Total revenue  : ${total:,.0f}")
print(f"   Top-3 share    : {conc_pct:.1f}%  [{risk} RISK]")
print()
for name, rev in top3:
    print(f"   {name:<20}  ${rev:>8,.0f}  ({rev/total*100:.1f}%)")

---
## 7. Operational Efficiency

### 7.1 Gross Margin by Product

In [ ]:
COGS_PCT = {"Filly Suite": 0.28, "Aegis Security": 0.35, "Filly API": 0.18, "Other": 0.40}

gm_data = []
for p in products:
    cogs = p["amount"] * COGS_PCT.get(p["name"], 0.30)
    gp   = p["amount"] - cogs
    gm_data.append({
        "product": p["name"], "revenue": p["amount"],
        "cogs": round(cogs), "gross_profit": round(gp),
        "margin_pct": round(gp / p["amount"] * 100, 1)
    })

print(f"{'Product':<18} {'Revenue':>9} {'COGS':>8} {'Gross Profit':>13} {'Margin':>7}")
print("-" * 58)
for d in gm_data:
    print(f"{d['product']:<18} ${d['revenue']:>8,} ${d['cogs']:>7,} ${d['gross_profit']:>12,} {d['margin_pct']:>6.1f}%")

prod_names = [d["product"]      for d in gm_data]
rev_vals   = [d["revenue"]      for d in gm_data]
cogs_vals  = [d["cogs"]         for d in gm_data]
gp_vals    = [d["gross_profit"] for d in gm_data]
xi = np.arange(len(prod_names)); w = 0.28

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(xi - w, rev_vals,  w, color=ACCENT3, alpha=0.85, label="Revenue")
ax.bar(xi,     cogs_vals, w, color=ACCENT2, alpha=0.85, label="COGS")
ax.bar(xi + w, gp_vals,   w, color=ACCENT1, alpha=0.85, label="Gross Profit")
for i, d in enumerate(gm_data):
    ax.text(i + w, d["gross_profit"] + 200, f"{d['margin_pct']}%", ha="center", fontsize=8, color=ACCENT1)
ax.set_title("Gross Margin by Product Line", fontsize=13, color=TEXT, pad=16)
ax.set_xticks(xi); ax.set_xticklabels(prod_names)
ax.set_ylabel("USD"); ax.yaxis.grid(True); ax.set_axisbelow(True); ax.legend(framealpha=0)
plt.tight_layout(); plt.show()

### 7.2 Expense Breakdown

In [ ]:
EXPENSE_CATEGORIES = {"Payroll": 0.55, "Infrastructure": 0.12, "Marketing": 0.18, "G&A": 0.10, "Other": 0.05}
total_exp = metrics["expenses_ytd"]
breakdown = [{"category": c, "amount": round(total_exp * p), "share": round(p*100,1)} for c, p in EXPENSE_CATEGORIES.items()]

print(f"💼 Expense Breakdown  (Total: ${total_exp:,})")
for d in breakdown:
    bar = "█" * int(d["share"] / 2)
    print(f"   {d['category']:<16} ${d['amount']:>8,}  {d['share']:>5.1f}%  {bar}")

fig, ax = plt.subplots(figsize=(7, 7))
fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
colors_pie = [ACCENT1, ACCENT3, ACCENT4, ACCENT2, MUTED]
wedges, texts, autotexts = ax.pie(
    [d["amount"] for d in breakdown], labels=[d["category"] for d in breakdown],
    colors=colors_pie, autopct="%1.0f%%", startangle=90, pctdistance=0.78,
    wedgeprops={"width": 0.52, "edgecolor": BG, "linewidth": 2},
)
for t in texts:     t.set_color(TEXT); t.set_fontsize(11)
for at in autotexts: at.set_color(BG); at.set_fontsize(9); at.set_fontweight("bold")
ax.set_title("Expense Breakdown by Category", fontsize=13, color=TEXT, pad=20)
ax.text(0, 0, f"${total_exp/1_000:.0f}K", ha="center", va="center", fontsize=18, fontweight="bold", color=TEXT)
plt.tight_layout(); plt.show()

### 7.3 Headcount Efficiency

In [ ]:
HEADCOUNT = 12
BENCHMARK = 150_000
rev_per_head = metrics["revenue_ytd"] / HEADCOUNT
vs_benchmark = (rev_per_head / BENCHMARK - 1) * 100
hc_health    = "EXCELLENT" if rev_per_head >= BENCHMARK*1.5 else "GOOD" if rev_per_head >= BENCHMARK else "CAUTION" if rev_per_head >= BENCHMARK*0.6 else "POOR"

print(f"👥 Headcount Efficiency")
print(f"   Revenue YTD      : ${metrics['revenue_ytd']:,}")
print(f"   Headcount        : {HEADCOUNT}")
print(f"   Revenue per head : ${rev_per_head:,.0f}")
print(f"   vs benchmark     : {vs_benchmark:+.1f}%  (benchmark: ${BENCHMARK:,})")
print(f"   Status           : {hc_health}")

---
## 8. Forecasting

### 8.1 3-Scenario Revenue Forecast

In [ ]:
SCENARIOS = {
    "bear": {"growth_rate": 0.03, "label": "Bear 🐻", "color": ACCENT2},
    "base": {"growth_rate": 0.07, "label": "Base 📊", "color": ACCENT3},
    "bull": {"growth_rate": 0.12, "label": "Bull 🚀", "color": ACCENT1},
}
FORECAST_MONTHS = 12

forecast = {}
for key, cfg in SCENARIOS.items():
    series, m = [], mrr
    for i in range(1, FORECAST_MONTHS + 1):
        m = m * (1 + cfg["growth_rate"])
        series.append({"month": i, "mrr": round(m)})
    forecast[key] = {**cfg, "series": series, "final_mrr": series[-1]["mrr"], "final_arr": series[-1]["mrr"] * 12}

for key, sc in forecast.items():
    print(f"   {sc['label']:<10}  Final MRR: ${sc['final_mrr']:>9,}  ARR: ${sc['final_arr']:>12,}")

hist_vals = [m["revenue"] / 1_000 for m in monthly]
hist_x    = list(range(-len(hist_vals), 0))
fig, ax   = plt.subplots(figsize=(13, 6))
ax.plot(hist_x, hist_vals, color=TEXT, linewidth=1.8, linestyle="--", label="Historical", alpha=0.6)
for key, sc in forecast.items():
    fx  = list(range(1, FORECAST_MONTHS + 1))
    fmr = [m["mrr"] / 1_000 for m in sc["series"]]
    ax.plot(fx, fmr, color=sc["color"], linewidth=2.2, label=sc["label"])
    ax.fill_between(fx, fmr, alpha=0.08, color=sc["color"])
    ax.annotate(f"${sc['final_arr']/1_000_000:.2f}M ARR", xy=(fx[-1], fmr[-1]),
                xytext=(5, 0), textcoords="offset points", color=sc["color"], fontsize=8)
ax.axvline(0, color=MUTED, linewidth=0.8, linestyle=":")
ax.set_title("12-Month Revenue Forecast — Bear / Base / Bull", fontsize=13, color=TEXT, pad=16)
ax.set_xlabel("Months from today"); ax.set_ylabel("MRR (USD '000)")
ax.yaxis.grid(True); ax.set_axisbelow(True); ax.legend(framealpha=0)
plt.tight_layout(); plt.show()

### 8.2 Monte Carlo Burn Simulation

In [ ]:
rng = np.random.default_rng(42)
SIMULATIONS     = 1_000
HORIZON_MONTHS  = 24
BURN_VOLATILITY = 0.10

runways = []
for _ in range(SIMULATIONS):
    cash, ran_out = CASH_BALANCE, HORIZON_MONTHS
    for m in range(HORIZON_MONTHS):
        burn = MONTHLY_BURN * (1 + rng.normal(0, BURN_VOLATILITY))
        cash -= max(burn, 0)
        if cash <= 0:
            ran_out = m + 1
            break
    runways.append(ran_out)

runways = np.array(runways)
p10, p50, p90 = int(np.percentile(runways, 10)), int(np.percentile(runways, 50)), int(np.percentile(runways, 90))
pct_survive   = round(float(np.mean(runways == HORIZON_MONTHS)) * 100, 1)

print(f"🎲 Monte Carlo Burn Simulation ({SIMULATIONS:,} runs, {HORIZON_MONTHS}-month horizon)")
print(f"   P10 pessimistic : {p10} months")
print(f"   P50 median      : {p50} months")
print(f"   P90 optimistic  : {p90} months")
print(f"   Survive horizon : {pct_survive}%")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(runways, bins=40, color=ACCENT3, alpha=0.75, edgecolor=BG)
ax.axvline(p10, color=ACCENT2, linewidth=1.5, linestyle="--", label=f"P10: {p10} mo")
ax.axvline(p50, color=ACCENT4, linewidth=1.8,                 label=f"P50: {p50} mo")
ax.axvline(p90, color=ACCENT1, linewidth=1.5, linestyle="--", label=f"P90: {p90} mo")
ax.set_title(f"Monte Carlo Burn Simulation (n={SIMULATIONS:,})", fontsize=13, color=TEXT, pad=16)
ax.set_xlabel("Months until cash runs out"); ax.set_ylabel("Simulations")
ax.yaxis.grid(True); ax.set_axisbelow(True); ax.legend(framealpha=0)
plt.tight_layout(); plt.show()

### 8.3 CAC Trend & Extrapolation

In [ ]:
cac_hist = [m["cac"] for m in monthly]
x_hist   = np.arange(len(cac_hist))
slope, intercept = np.polyfit(x_hist, cac_hist, 1)

FORECAST_CAC_MONTHS = 6
last_date = datetime.strptime(monthly[-1]["month"], "%b %Y")
cac_forecast = []
for i in range(1, FORECAST_CAC_MONTHS + 1):
    y, mo = divmod(last_date.month - 1 + i, 12)
    next_mo = datetime(last_date.year + y, mo + 1, 1)
    cac_forecast.append({
        "month": next_mo.strftime("%b %Y"),
        "cac":   round(max(0, slope * (len(cac_hist) + i) + intercept), 2)
    })

all_labels = [m["month"] for m in monthly] + [f["month"] for f in cac_forecast]
all_cac    = cac_hist + [f["cac"] for f in cac_forecast]
split      = len(cac_hist)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(range(split), cac_hist, color=ACCENT4, linewidth=2.2,
        marker="o", markersize=5, markerfacecolor=BG, label="Actual")
ax.plot(range(split - 1, len(all_cac)), [cac_hist[-1]] + [f["cac"] for f in cac_forecast],
        color=ACCENT2, linewidth=2, linestyle="--",
        marker="s", markersize=4, markerfacecolor=BG, label="Forecast")
ax.axvline(split - 1, color=MUTED, linewidth=0.8, linestyle=":")
ax.set_xticks(range(len(all_labels))); ax.set_xticklabels(all_labels, rotation=35, ha="right", fontsize=8)
ax.set_title("CAC Trend & 6-Month Extrapolation", fontsize=13, color=TEXT, pad=16)
ax.set_ylabel("CAC (USD)"); ax.yaxis.grid(True); ax.set_axisbelow(True); ax.legend(framealpha=0)
plt.tight_layout(); plt.show()

print(f"\n📈 CAC Forecast")
for f in cac_forecast:
    print(f"   {f['month']}  →  ${f['cac']:,.2f}")

---
## 9. Executive Summary

In [ ]:
print("=" * 55)
print("  STARTUP.OS — Executive Summary")
print("=" * 55)
print(f"  Revenue YTD      : ${metrics['revenue_ytd']:>12,}")
print(f"  Net Profit YTD   : ${metrics['profit_ytd']:>12,}  ({metrics['profit_margin']:.0f}% margin)")
print(f"  Expenses YTD     : ${metrics['expenses_ytd']:>12,}")
print(f"  MRR              : ${mrr:>12,}")
print(f"  ARR              : ${mrr*12:>12,}")
print(f"  CAC              : ${metrics['cac']:>12,}")
print(f"  LTV:CAC          : {ratio:>11.1f}x")
print(f"  Payback period   : {payback_months:>9.1f} months")
print(f"  Runway           : {months_runway:>9.1f} months")
print(f"  NRR              : {nrr:>10.1f}%")
print(f"  Annual churn     : {annual_churn*100:>10.1f}%")
print("=" * 55)
print("  All figures simulated for demo purposes.")
print("=" * 55)